##### **Processing prev_data**

In [1]:
def fetch_json_data(file_path, output_file_path="cust_data.json"):
    """
    Fetches recipe data from a local JSON file, processes it, and returns only the relevant fields.
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)

        all_recipes = []

        # Iterate over each entry in the JSON file
        for entry in data:
            # Skip if entry is not a dictionary
            if not isinstance(entry, dict):
                print(f"Skipping non-dictionary entry: {entry}")
                continue

            recipe_info_list = entry.get("recipe_info", [])  # Ensure it's a list

            # Ensure recipe_info_list is actually a list before proceeding
            if not isinstance(recipe_info_list, list):
                print(f"Skipping entry due to unexpected format: {entry}")
                continue

            # Extract delivery date - now much simpler
            delivery_date = None
            entry_id = entry.get("_id", {})
            if isinstance(entry_id, dict):
                delivery_date = entry_id.get("delivery_date")
                # You might want to validate the date format here if needed

            # Iterate through each recipe_info dictionary in the list
            for recipe_info in recipe_info_list:
                # Skip if recipe_info is not a dictionary
                if not isinstance(recipe_info, dict):
                    print(f"Skipping non-dictionary recipe_info: {recipe_info}")
                    continue

                # Extract ingredients and merge with variant ingredients
                ingredients = recipe_info.get("ingredients", []) or []  # Ensure it's always a list
                variant_ingredients = []

                # Process the variants if available
                variants = recipe_info.get("variants", {})
                if isinstance(variants, dict):  # Ensure "variants" is a dictionary
                    variant_ingredients = variants.get("variant_ingredients", []) or []  # Ensure it's always a list

                # Merge ingredients from both the recipe_info and variants
                all_ingredients = list(set(ingredients + variant_ingredients))

                # Prepare the recipe data with only the necessary fields
                recipe = {
                    "dish_name": recipe_info.get("dish_name"),
                    "meal_category": recipe_info.get("meal_category"),
                    "description": recipe_info.get("description"),
                    "cuisine": recipe_info.get("cuisine"),
                    "ingredients": all_ingredients,  # Merged ingredients
                    "allergens_contain": recipe_info.get("allergens_contain", []),
                    "meal_type": entry.get("meal_type"),
                    "spice_level": recipe_info.get("spice_level", ""),
                    "is_auto_select": recipe_info.get("is_auto_select"),
                    "rating": recipe_info.get("rating") if "rating" in recipe_info else None,
                    "delivery_date": delivery_date  # Use the date string directly
                }

                # Add the processed recipe data to the list
                all_recipes.append(recipe)

        # If an output file path is provided, save the processed data to that file
        if output_file_path and all_recipes:
            with open(output_file_path, 'w') as output_file:
                json.dump(all_recipes, output_file, indent=4)
            print(f"Processed data saved to {output_file_path}")

        return all_recipes

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error processing file {file_path}: {str(e)}")
        return []

In [2]:
import os
import json
# Define the input and output directories
input_directory = "user_data/prev_data"
output_directory = "user_data/processed_files"

# Ensure output directory exists
os.makedirs(output_directory, exist_ok=True)

# Get all JSON files in the input directory
input_file_paths = [os.path.join(input_directory, file) for file in os.listdir(input_directory) if file.endswith(".json")]

# Loop through each input file and process it
for input_file_path in input_file_paths:
    file_name = os.path.basename(input_file_path)
    output_file_path = os.path.join(output_directory, f"processed_{file_name}")
    
    processed_data = fetch_json_data(input_file_path, output_file_path)
    
    if processed_data:
        print(f"Processed data for {file_name} saved to {output_file_path}")
    else:
        print(f"Failed to process data for {file_name}.")


Processed data saved to user_data/processed_files/processed_64ea0ff65d497c8a38e10c6c.json
Processed data for 64ea0ff65d497c8a38e10c6c.json saved to user_data/processed_files/processed_64ea0ff65d497c8a38e10c6c.json
Processed data saved to user_data/processed_files/processed_659af0c214c28bbc7574d932.json
Processed data for 659af0c214c28bbc7574d932.json saved to user_data/processed_files/processed_659af0c214c28bbc7574d932.json
Processed data saved to user_data/processed_files/processed_6652fec04df8c59ae54b5988.json
Processed data for 6652fec04df8c59ae54b5988.json saved to user_data/processed_files/processed_6652fec04df8c59ae54b5988.json
Processed data saved to user_data/processed_files/processed_67061ae623ead763336b938f.json
Processed data for 67061ae623ead763336b938f.json saved to user_data/processed_files/processed_67061ae623ead763336b938f.json
Processed data saved to user_data/processed_files/processed_67b0ca24bb439e10a7e42e23.json
Processed data for 67b0ca24bb439e10a7e42e23.json saved

##### **Pinecone query generation**

In [3]:
import pandas as pd

import json
from IPython.display import display, Markdown

def process_and_analyze_json(output_file_path="processed_data.json"):
    """
    Process and analyze the recipe data from a JSON file.
    It will:
    - Extract and merge the ingredients.
    - Analyze top cuisines, spice levels, and user selections.
    - Display visualizations like pie charts.
    - List all user-rated recipes sorted from highest to lowest rating.
    """
    try:
        # Load the processed JSON data directly
        with open(output_file_path, "r") as file:
            recipes = json.load(file)

        # Convert the processed data into a DataFrame
        df = pd.DataFrame(recipes)

        # Check if 'cuisine' column exists
        if 'cuisine' not in df.columns:
            raise KeyError("Missing 'cuisine' column in the dataset")

        # Identify the top cuisine preference
        top_cuisine = df['cuisine'].mode()
        # print(f"Top Cuisine: {top_cuisine}")
        
        # Identify the user's most preferred spice level
        top_spice_level = df['spice_level'].mode()[0]
        # print(f"Top Spice Level: {top_spice_level}")

        # No longer filtering by `is_auto_select`
        user_selected_meals = df.copy()  

        # Display a summary of all meals
        user_selected_meals_summary = user_selected_meals.describe(include='object')
        # print("Summary of All Selected Meals:")
        # display(user_selected_meals_summary)

        # Get the most frequent dish names
        top_dish_names = user_selected_meals['dish_name'].value_counts().reset_index()
        top_n = 5
        # display(Markdown("### Most Frequently Selected Dishes:"))
        # display(top_dish_names.head(top_n))

        # Get the count of each cuisine selected by the user
        cuisine_counts = user_selected_meals['cuisine'].value_counts().reset_index()
        cuisine_counts.columns = ["Cuisine", "Count"]
        # display(Markdown("### Total Cuisine Counts Across all Weeks :"))
        # display(cuisine_counts)

        # Display all user-rated recipes sorted from highest to lowest rating
        if 'rating' in df.columns:
            top_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=False)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(top_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        if 'rating' in df.columns:
            least_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=True)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(least_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        # Pie Chart for Cuisine Preferences
        # sns.set(style="darkgrid")
        # plt.style.use('dark_background')

        # plt.figure(figsize=(4, 4))  
        # plt.pie(
        #     cuisine_counts['Count'], 
        #     labels=cuisine_counts['Cuisine'], 
        #     autopct='%1.1f%%',  
        #     colors=sns.color_palette("tab20", len(cuisine_counts)), 
        #     startangle=90, 
        #     wedgeprops={'edgecolor': 'none'},  
        #     labeldistance=1.1,  
        #     pctdistance=0.85  
        # )
        # plt.title('Cuisines Selected by User', fontsize=8, color='white')  
        # plt.ylabel('')
        # plt.gcf().patch.set_alpha(0)
        # # display(Markdown("### Cuisine Preference Across all Weeks :"))
        # plt.show()

        # Weekly Cuisine Preferences Analysis
        if 'delivery_date' in df.columns:
            df['delivery_date'] = df['delivery_date'].apply(lambda x: str(x) if isinstance(x, dict) else x)
            df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
        
        min_date = df['delivery_date'].min()
        df['week_number'] = df['delivery_date'].apply(lambda x: (x - min_date).days // 7 + 1)

        cuisine_weekly_counts = df.groupby(['week_number', 'cuisine']).size().reset_index(name='count')
        pivot_table = cuisine_weekly_counts.pivot(index="week_number", columns="cuisine", values="count").fillna(0)
        cuisine_order = cuisine_weekly_counts.groupby("cuisine")["count"].sum().sort_values()
        pivot_table = pivot_table[cuisine_order.index]

        # display(Markdown("### Cuisine Preferences over Weeks :"))
        # display(pivot_table)

        # Generate structured query based on analysis
        top_cuisines = ', '.join(df['cuisine'].mode())  # Get top cuisine(s)
        top_spice_level = df['spice_level'].mode()[0]  # Get top spice level
        # Ensure the first column contains dish names
        top_dishes = ', '.join(top_dish_names.iloc[:6, 0])  # ✅ Corrected way

        # Find highest-rated dishes (handling multiple)
        if 'rating' in df.columns and not df['rating'].isna().all():
            max_rating = df['rating'].max()  # Get the highest rating
            top_rated_dishes = df[df['rating'] == max_rating]['dish_name'].unique()  # Get unique top-rated dishes
            top_rated_dishes = ', '.join(top_rated_dishes[:5])  # Limit to top 5 for readability
        else:
            top_rated_dishes = None
        # Find dishes rated less than 3 (User Dislikes)
        user_dislikes = None
        if 'rating' in df.columns and not df['rating'].isna().all():
            disliked_dishes = df[df['rating'] < 3]['dish_name'].unique()  # Get unique low-rated dishes
            user_dislikes = ', '.join(disliked_dishes[:5])  # Limit to 5 for readability

        # Build the query string
        query = f"Spice Level: {top_spice_level}, Cuisine: {top_cuisines}. Popular Dishes: {top_dishes}"

        if top_rated_dishes:
            query += f". Highest Rated Dishes: {top_rated_dishes}"

        # print("Generated Query:")
        # print(query)

        return df, query, user_dislikes  # Return DataFrame and Query


    except FileNotFoundError:
        print(f"Error: The file '{output_file_path}' was not found.")
        return []
    except KeyError as e:
        print(f"Error: {str(e)}")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{output_file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error: {str(e)}")
        return []

In [4]:
import os
import glob

# Folder containing the JSON files
folder_path = "user_data/processed_files/"

# Get all JSON files in the folder
json_files = glob.glob(os.path.join(folder_path, "*.json"))
print(f"Found {len(json_files)} JSON files in {folder_path}")
# Process each file
for file_path in json_files:
    print(f"\nProcessing: {file_path}")
    df, query, user_dislikes = process_and_analyze_json(file_path)  # Call function for each file
    print(f"Query for {file_path}:\n{query}\n")


Found 100 JSON files in user_data/processed_files/

Processing: user_data/processed_files/processed_67961709889e8bdf42039a1b.json
Query for user_data/processed_files/processed_67961709889e8bdf42039a1b.json:
Spice Level: Low, Cuisine: European. Popular Dishes: Shawarma Bowl , Mushroom Spinach & Cheddar Omelette, Coconut Lime Zest, Chocolate Muffin , Cajun Roasted Chicken Sandwich, Greek Yogurt. Highest Rated Dishes: Gruyere Omelette


Processing: user_data/processed_files/processed_651400dd5f57cba5d84a8d7d.json
Query for user_data/processed_files/processed_651400dd5f57cba5d84a8d7d.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Crudites & Sour Cream Dip , Lemon Quinoa with Roasted Protein, Mushroom Penne, Creamy Tomato Tagliatelle , Kalamata Olives & Parmesan Cheese, Sticky Mango & Coconut Rice Pudding 


Processing: user_data/processed_files/processed_66a3bcf4430fb89a1a6da0ab.json
Query for user_data/processed_files/processed_66a3bcf4430fb89a1a6da0ab.json:
Spice Level: 

### **CSV collection**

In [11]:
import os
import json
import pandas as pd
import glob

def clean_string(value):
    """Helper function to clean strings and strip whitespace"""
    if isinstance(value, str):
        return value.strip()
    return value

def clean_string_list(value_list):
    """Helper function to clean lists of strings and strip whitespace"""
    if isinstance(value_list, (list, set)):
        return [item.strip() for item in value_list if isinstance(item, str)]
    return value_list

def extract_user_preferences(csv_path="csv/auto_test.csv", json_folder="user_data/processed_files/"):
    """
    Extracts user preferences from CSV and JSON files and returns a structured dictionary.
    Trims whitespace from all string fields and lists.
    """
    # Load CSV
    df_csv = pd.read_csv(csv_path)

    # Dictionary to store user preferences
    user_preferences = {}

    # Get all JSON files in the folder
    json_files = glob.glob(os.path.join(json_folder, "*.json"))

    # Process each matching file
    for file_path in json_files:
        # Extract user_id from file name and clean it
        user_id = clean_string(os.path.basename(file_path).replace("processed_", "").replace(".json", ""))

        # Find matching row in CSV
        matching_row = df_csv[df_csv['customer_id'] == user_id]

        if not matching_row.empty:
            print(f"\nProcessing: {file_path}")

            # Extract values from CSV row
            row = matching_row.iloc[0]

            # Avoid ingredients (default to empty set if missing)
            user_avoid_ingredients = set(
                clean_string(ingredient) 
                for ingredient in str(row["avoid_ingredient"]).split(',') 
                if pd.notna(row["avoid_ingredient"])
            ) if pd.notna(row["avoid_ingredient"]) else set()

            # Protein category (default to empty string if missing)
            protein_category = clean_string(row['protein_category']) if pd.notna(row['protein_category']) else ""

            # Size (default to empty string if missing)
            size = clean_string(row["variant_size"]) if pd.notna(row["variant_size"]) else ""

            # Meal type as a set (default to empty set if missing)
            meal_types = set(
                clean_string(meal_type) 
                for meal_type in str(row["plan"]).split(',') 
                if pd.notna(row["plan"])
            ) if pd.notna(row["plan"]) else set()


            # Number of days (default to None if missing)
            num_days = int(row["num_days"]) if pd.notna(row["num_days"]) else None

            # Process JSON file
            df, query, user_dislikes = process_and_analyze_json(file_path)
            
            # Clean the query results
            cleaned_query = clean_string(query)
            user_dislikes = clean_string_list(user_dislikes)

            # Store results in a dictionary with cleaned values
            user_preferences[user_id] = {
                "user_pref": clean_string(cleaned_query.split("Popular Dishes: ")[1]) 
                           if "Popular Dishes: " in cleaned_query else "",
                "user_likes": f"{clean_string(cleaned_query.split('Cuisine: ')[1].split('.')[0])} cuisine" 
                             if "Cuisine: " in cleaned_query else "",
                "user_avoid_ingredients": user_avoid_ingredients,
                "size": size,
                "protein_option": "",  # Always default to empty string
                "protein_category": protein_category,
                "meal_types": meal_types,
                "num_days": num_days,  # Added num_days field
                "query": cleaned_query,
                "user_dislikes": user_dislikes
            }

    return user_preferences

### **Recommendations**

In [6]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_avoid_ingredients, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_avoid_ingredients)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_avoid_ingredients)}  # Exclude recipes containing allergens
    }
    # # Add `protein_category` filter only if it's not empty
    # if protein_category:
    #     pinecone_filter["protein_category"] = {"$eq": protein_category}


    # Fetch documents using similarity_search
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k * 5 ,  # Get extra to account for duplicates
        filter=pinecone_filter
    )

    # Merge recipes by recipe_id and combine protein_options
    merged_recipes = {}

    
    for doc in docs:
        metadata = doc.metadata
        recipe_id = metadata.get("recipe_id")
        
        # Skip if no recipe_id exists
        if not recipe_id:
            continue
        
        # Initialize new entry if recipe_id not seen
        if recipe_id not in merged_recipes:
            merged_recipes[recipe_id] = {
                **metadata,  # Copy all metadata
                "protein_option": {metadata.get("protein_option")}  # Start as set
            }
        else:
            # Merge protein_options
            existing = merged_recipes[recipe_id]
            new_protein = metadata.get("protein_option")
            if new_protein:
                existing["protein_option"].add(new_protein)
    
    # Prepare final output (convert sets to lists)
    final_recipes = [
        {
            **data,
            "protein_option": list(data["protein_option"]) if data["protein_option"] else []
        }
        for data in merged_recipes.values()
    ]
    # print("final",final_recipes[:top_k])
    return final_recipes[:top_k]

# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(merged_recipes, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types, num_days):

        # Normalize meal types by stripping whitespace and converting to lowercase
    normalized_meal_types = {meal.strip().lower() for meal in meal_types}
    
    # Define the standard meal type order (customize as needed)
    STANDARD_ORDER = ['morning_snack', 'breakfast', 'lunch', 'dinner', 'evening_snack']
    
    # Filter and order the meal types based on standard order
    ordered_meal_types = [meal for meal in STANDARD_ORDER 
                         if meal in normalized_meal_types]
    
    prompt = f"""Generate a {num_days} day meal plan in JSON format using ONLY the provided recipes. Follow these rules exactly:

1. Recipe Usage:
- Use recipes exactly as provided - do not modify or create new ones
- Format each meal as: "<Dish Name> - <Selected Protein> - <Cuisine> - <Dish Type>"
- Use ONLY the dish_type field for the last component (never meal_category)
- For recipes with multiple protein options:
    * Ensure protein variety across the week (don't serve chicken 3 days in a row)

3. Meal Diversity:
- Alternate between:
  * Light vs heavy meals (e.g. salad → hearty stew)
  * Different cuisines (don't repeat back-to-back)
  * Cooking methods (grilled, baked, fried, etc.)
- Ensure no two consecutive meals have:
  * The same primary ingredient
  * Similar textures/flavor profiles

2. Meal Assignment:
- Never repeat recipes before all are used once.
- If recipes are less than required then you may repeat them.
- Only consider the recipes provided for meal plan not even the ones that are present in the Likes and preferences of the user.
- Fill all selected meal slots - no empty values
- **Strictly follow meal categories:**
    * Breakfast: only 'breakfast' recipes having meal_category as breakfast
    * Lunch/Dinner: only 'meal' recipes having meal_category as meal
    * evening_snavk/morning_snack: only 'snack' recipes i.e. recipes having meal_category as snack
- Include only these meal types: {meal_types}

3. Daily Structure:
- You MUST include these meal types in EXACTLY this order: {ordered_meal_types}
- Never skip or rearrange these meal types
- Never include meal types not in this list
- Maintain consistent meal types across all days

Output Format: Present the meal plan as a JSON object where each day contains meal types as keys and the formatted meal string as values, like this example for Monday: {{\"Monday\": {{\"breakfast\": \"Dish Name - Protein - Cuisine - Dish Type\", \"lunch\": \"...\"}}}}User Preferences:
- Allergens: {user_avoid_ingredients}
- Likes: {user_likes}
- Dislikes: {user_dislikes}
- Preferred Dishes: {user_pref}
- Selected Meal Types: {meal_types}

Available Recipes:"""
    

    for i, recipe in enumerate(merged_recipes, 1):
        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {recipe.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {recipe.get('description', 'No description')}\n"
        prompt += f" Protein Options: {', '.join(recipe.get('protein_option', []))}\n"  # Changed to handle list
        prompt += f" Dish Type: {', '.join(recipe.get('dish_type', []))}\n"  # Changed to handle list
        prompt += f" Ingredients: {', '.join(recipe.get('ingredients', []))}\n"
        prompt += f" Spice Level: {recipe.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {recipe.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {recipe.get('meal_category', 'Unknown')}\n"
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref, size, protein_option, protein_category, meal_types, num_days):
    # Define mapping of meal types to their respective counts
    meal_counts = {
        "breakfast": num_days,
        "snack": num_days,  # Each snack type (morning/evening) adds 8
        "meal": 0    # Lunch and Dinner are combined into "meal"
    }

    # Initialize counts
    total_snack_count = 0
    total_meal_count = 0
    fetched_recipes = {}

    # Calculate needed recipes
    if "morning_snack" in meal_types or "evening_snack" in meal_types:
        total_snack_count = meal_counts["snack"] * sum(1 for meal in meal_types if "snack" in meal)

    if "lunch" in meal_types:
        total_meal_count += num_days
    if "dinner" in meal_types:
        total_meal_count += num_days

    # Initial fetch (with allergens)
    for meal_type in meal_types:
        count = meal_counts.get(meal_type, 0)
        if count > 0:
            meal_size = "standard" if meal_type in ["breakfast", "snack"] else size
            fetched_recipes[meal_type] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, 
                meal_type, meal_size, "", protein_category, (count+3)
            )

    if total_meal_count > 0:
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "meal", size, protein_option, protein_category, (total_meal_count+3)
        )

    if total_snack_count > 0:
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query,
            "snack", "standard", "", protein_category, (total_snack_count+3)
        )

    # # Calculate expected minimums
    # expected_breakfast = max(0, meal_counts.get("breakfast", 0) - 4) if "breakfast" in meal_types else 0
    # expected_snack = max(0, total_snack_count - (3 if total_snack_count == 8 else 6 if total_snack_count == 16 else 0)) if any("snack" in mt for mt in meal_types) else 0
    # expected_meal = max(0, total_meal_count - (2 if total_meal_count == 8 else 7 if total_meal_count == 20 else 0)) if ("lunch" in meal_types or "dinner" in meal_types) else 0
    # print("fetched",fetched_recipes)    
    
    expected_breakfast = max((min(num_days, 4) + min(1, num_days)) - (0 if num_days <= 2 else 1 if num_days <= 5 else min(2, (min(num_days, 4) + min(1, num_days)) // 3)), 1) if "breakfast" in meal_types else 0
    expected_snack = max(total_snack_count - (0 if num_days <= 3 else 1 if num_days <= 5 else min(2, total_snack_count // 3)), 1) if any("snack" in mt for mt in meal_types) else 0
    expected_meal = max(total_meal_count - (0 if num_days <= 3 else 1 if num_days <= 5 else min(2, total_meal_count // 3)), 1) if ("lunch" in meal_types or "dinner" in meal_types) else 0    
    
    # print("fetched",fetched_recipes)


    # Selective retry (only replaces deficient categories)
    if "breakfast" in meal_types and len(fetched_recipes.get("breakfast", [])) < expected_breakfast:
        print(f"Breakfast shortage ({len(fetched_recipes.get('breakfast', []))}/{expected_breakfast}), retrying without filters...")
        fetched_recipes["breakfast"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "breakfast", "standard", "", protein_category, meal_counts["breakfast"]+3
        )

    if any("snack" in mt for mt in meal_types) and len(fetched_recipes.get("snack", [])) < expected_snack:
        print(f"Snack shortage ({len(fetched_recipes.get('snack', []))}/{expected_snack}), retrying without filters...")
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "snack", "standard", "", protein_category, total_snack_count+3
        )

    if ("lunch" in meal_types or "dinner" in meal_types) and len(fetched_recipes.get("meal", [])) < expected_meal:
        print(f"Meal shortage ({len(fetched_recipes.get('meal', []))}/{expected_meal}), retrying without filters...")
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, set(), user_dislikes, query,
            "meal", size, protein_option, protein_category, total_meal_count+3
        )

    # if ("lunch" in meal_types or "dinner" in meal_types) and len(fetched_recipes.get("meal", [])) < expected_meal:
    #     current_count = len(fetched_recipes.get("meal", []))
    #     shortfall = expected_meal - current_count  # Calculate the shortfall
    #     print(f"Meal shortage ({current_count}/{expected_meal}), retrying without filters...")
    #     additional_recipes = filter_recipes(
    #         vectorstore, set(), user_dislikes, query,
    #         "meal", size, protein_option, protein_category, shortfall
    #     )
    # fetched_recipes["meal"].extend(additional_recipes)  # Append to existing recipes

    # print("fetched",fetched_recipes)

    # Debug output
    print("\nFinal recipe counts:")
    for category in ["breakfast", "snack", "meal"]:
        if category in fetched_recipes:
            print(f"{category.capitalize()}: {len(fetched_recipes[category])}")

    # Generate meal plan
    final_docs = [recipe for recipes in fetched_recipes.values() for recipe in recipes]
    final_prompt = format_meal_plan_prompt(final_docs, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types, num_days)
    meal_plan = generate_response(final_prompt)

    return meal_plan, final_docs

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_avoid_ingredients = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref)

/home/hello/sb/recommendation_del/venv/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/tmp/ipykernel_8920/3662654750.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/tmp/ipykernel_8920/3662654750.py:28: LangChainDeprecationWarning: The class `Pinecone` was deprecated in LangChain 0.0.18 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-pin

In [12]:
import time

# Call function to extract user preferences
user_preferences = extract_user_preferences()

# Iterate over all users and generate meal plans
for user_id, prefs in user_preferences.items():
    print(f"\nGenerating meal plan for User ID: {user_id}")

    # Extract user details
    user_avoid_ingredients = prefs["user_avoid_ingredients"] if prefs["user_avoid_ingredients"] else {}  
    user_dislikes = prefs["user_dislikes"] if prefs["user_dislikes"] else {}  
    query = prefs["query"]
    user_likes = prefs["user_likes"]
    user_pref = prefs["user_pref"]
    size = prefs["size"].lower() if prefs["size"] else ""  # Convert to lowercase
    protein_option = prefs["protein_option"] if prefs["protein_option"] else ""  
    protein_category = prefs["protein_category"].lower() if prefs["protein_category"] else ""  # Convert to lowercase
    meal_types = prefs["meal_types"] if prefs["meal_types"] else set()  
    # Normalize meal_types by stripping spaces
    meal_types = {meal.strip() for meal in meal_types}
    num_days = prefs["num_days"] if "num_days" in prefs else 6  # Default to 7 days if not specified
    
    # Generate meal plan
    meal_plan, final_docs = generate_meal_plan(
        vectorstore,
        user_avoid_ingredients,
        user_dislikes,
        query,
        user_likes,
        user_pref,
        size,
        protein_option,
        protein_category,
        meal_types,
        num_days=num_days
    )

    # Print the meal plan
    print(f"\nMeal Plan for User ID: {user_id}:\n{meal_plan}\n")
    time.sleep(4)  # Maintains RPM limit



Processing: user_data/processed_files/processed_67961709889e8bdf42039a1b.json

Processing: user_data/processed_files/processed_651400dd5f57cba5d84a8d7d.json

Processing: user_data/processed_files/processed_66a3bcf4430fb89a1a6da0ab.json

Processing: user_data/processed_files/processed_64dde7c55d497c8a38be700d.json

Processing: user_data/processed_files/processed_678b4379c954b8182666bfb7.json

Processing: user_data/processed_files/processed_67e91b922e41aa516fa720c8.json

Processing: user_data/processed_files/processed_67ed63862e41aa516ffd93f1.json

Processing: user_data/processed_files/processed_651437cfd838112c9853c1f1.json

Processing: user_data/processed_files/processed_679e294a9c40d6abe11128a5.json

Processing: user_data/processed_files/processed_64ea0ff65d497c8a38e10c6c.json

Processing: user_data/processed_files/processed_679cb995ed9daf35d79a116c.json

Processing: user_data/processed_files/processed_64ff5cafb78e129edca5bc95.json

Processing: user_data/processed_files/processed_673

In [10]:
user_preferences

{'67d2d1531d73f3185b8f1829': {'user_pref': 'Noodles Bolognese , Beef Stew with Sweet Potato Mash, Chimichurri Steak, Thai Mango Salad, Greek Chicken with Potato Wedges, Creamy Quinoa Bowl . Highest Rated Dishes: Beetroot & Quinoa Salad , Beef Stew with Sweet Potato Mash, Baked Protein & Mashed Potato, Butter Masala & Quinoa, Mushroom Meatballs & Mash',
  'user_likes': 'Mediterranean cuisine',
  'user_avoid_ingredients': {'Cod (white fish)',
   'Cod Fish',
   'Cuttlefish',
   'Fish Sauce',
   'Gochujang Paste',
   'Gochujang Sauce',
   'Local Wild Fish',
   'Nile Perch',
   'Salmon',
   'Sea Bass',
   'Shrimps',
   'Squid',
   'Tuna',
   'White Fish',
   'Worcestershire Sauce'},
  'size': 'Medium',
  'protein_option': '',
  'protein_category': 'low',
  'meal_types': {'dinner', 'lunch'},
  'num_days': 5,
  'query': 'Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Noodles Bolognese , Beef Stew with Sweet Potato Mash, Chimichurri Steak, Thai Mango Salad, Greek Chicken with Pot

In [15]:
import json
import csv
import re
from collections import defaultdict

def process_mealplan_text(mealplan_text):
    # Split the text into individual user sections
    user_sections = re.split(r'Generating meal plan for User ID:', mealplan_text)[1:]
    
    all_user_data = []
    meal_types = set()
    
    for section in user_sections:
        # Extract user ID from the first line
        user_id = section.split('\n')[0].strip()
        
        # Check for shortage (retrying without filters)
        has_shortage = "retrying without filters" in section.lower()
        
        # Find JSON part
        json_match = re.search(r'```json\n({.*?})\n```', section, re.DOTALL)
        if not json_match:
            continue
            
        try:
            weekly_meals = json.loads(json_match.group(1))
            
            # Determine if we're using "Day X" format or weekday names
            day_format = None
            for day in weekly_meals.keys():
                if day.startswith("Day "):
                    day_format = "day_x"
                    break
                elif day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                    day_format = "weekday"
                    break
            
            # Collect all meal types across all users
            for day_meals in weekly_meals.values():
                if isinstance(day_meals, dict):
                    meal_types.update(day_meals.keys())
                
            # Format user data
            user_data = {
                "user_id": user_id,
                "shortage": "yes" if has_shortage else "no"
            }
            
            if day_format == "weekday":
                # Handle weekday format
                for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                    if day in weekly_meals:
                        meals = []
                        for meal_type, meal_details in weekly_meals[day].items():
                            meals.append(f"{meal_type} - {meal_details}")
                        user_data[day] = ", ".join(meals)
                    else:
                        user_data[day] = ""
            else:
                # Handle Day X format - map to weekdays sequentially
                days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
                day_numbers = sorted([int(day.split()[1]) for day in weekly_meals.keys() if day.startswith("Day ")])
                
                for i, day_num in enumerate(day_numbers):
                    day_key = f"Day {day_num}"
                    if i < len(days_of_week) and day_key in weekly_meals:
                        weekday = days_of_week[i]
                        meals = []
                        for meal_type, meal_details in weekly_meals[day_key].items():
                            meals.append(f"{meal_type} - {meal_details}")
                        user_data[weekday] = ", ".join(meals)
                    else:
                        if i < len(days_of_week):
                            user_data[days_of_week[i]] = ""
                
                # Fill any remaining weekdays with empty strings
                for j in range(len(day_numbers), len(days_of_week)):
                    user_data[days_of_week[j]] = ""
            
            all_user_data.append(user_data)
            
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Error processing user {user_id}: {str(e)}")
            continue
    
    return all_user_data, sorted(meal_types)

def save_to_csv(user_data, output_file="all_users_mealplans.csv"):
    fieldnames = ["user_id", "shortage", "Monday", "Tuesday", "Wednesday", 
                 "Thursday", "Friday", "Saturday", "Sunday"]
    
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(user_data)
    
    print(f"Successfully saved meal plans for {len(user_data)} users to {output_file}")

# Example usage with a text file containing multiple users
def process_mealplan_file(input_file_path):
    with open(input_file_path, 'r') as file:
        mealplan_text = file.read()
    
    user_data, meal_types = process_mealplan_text(mealplan_text)
    save_to_csv(user_data)
    
    print(f"Detected meal types across all users: {', '.join(meal_types)}")

# Run the processor
process_mealplan_file("meal_plan_days.txt")  # Replace with your input file path

Successfully saved meal plans for 100 users to all_users_mealplans.csv
Detected meal types across all users: breakfast, dinner, evening_snack, lunch, morning_snack


In [17]:
import csv
from collections import defaultdict

def merge_meal_plans(mealplan_csv, customer_csv, output_csv):
    # Read meal plan data into a dictionary
    meal_plans = {}
    with open(mealplan_csv, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            meal_plans[row['user_id']] = {
                'Monday': row['Monday'],
                'Tuesday': row['Tuesday'],
                'Wednesday': row['Wednesday'],
                'Thursday': row['Thursday'],
                'Friday': row['Friday'],
                'Saturday': row['Saturday'],
                'Sunday': row['Sunday'],
                'shortage': row.get('shortage', 'no')  # Get shortage field or default to 'no'
            }
    
    # Read customer data and merge with meal plans
    merged_data = []
    with open(customer_csv, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            customer_id = row['customer_id']
            if customer_id in meal_plans:
                # Update the row with meal plan data
                row.update({
                    'monday': meal_plans[customer_id]['Monday'],
                    'tuesday': meal_plans[customer_id]['Tuesday'],
                    'wednesday': meal_plans[customer_id]['Wednesday'],
                    'thursday': meal_plans[customer_id]['Thursday'],
                    'friday': meal_plans[customer_id]['Friday'],
                    'saturday': meal_plans[customer_id]['Saturday'],
                    'sunday': meal_plans[customer_id]['Sunday'],
                    'shortage': meal_plans[customer_id]['shortage']  # Add shortage field
                })
            else:
                # Add empty meal plan fields if customer not found in meal plans
                row.update({
                    'monday': '',
                    'tuesday': '',
                    'wednesday': '',
                    'thursday': '',
                    'friday': '',
                    'saturday': '',
                    'sunday': '',
                    'shortage': 'no'  # Default to 'no' if no meal plan exists
                })
            merged_data.append(row)
    
    # Write merged data to new CSV
    fieldnames = [
        'is_veg', 'customer_id', 'avoid_ingredient', 'plan', 
        'variant_size', 'protein_category', 'customer_name',
        'monday', 'tuesday', 'wednesday', 'thursday', 
        'friday', 'saturday', 'sunday', 'shortage'  # Added shortage field
    ]
    
    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(merged_data)

# Example usage:
merge_meal_plans(
    'csv/all_users_mealplans.csv',      # Your meal plan CSV with user_id and days
    'csv/AutoSelectionData.csv',       # Your customer data CSV
    'csv/merged_output.csv'    # Output file
)

In [19]:
import csv
from io import StringIO

# Your multiline string (replace with your actual data)
multiline_string = """Found 100 JSON files in user_data/processed_files/

Processing: user_data/processed_files/processed_67961709889e8bdf42039a1b.json
Query for user_data/processed_files/processed_67961709889e8bdf42039a1b.json:
Spice Level: Low, Cuisine: European. Popular Dishes: Shawarma Bowl , Mushroom Spinach & Cheddar Omelette, Coconut Lime Zest, Chocolate Muffin , Cajun Roasted Chicken Sandwich, Greek Yogurt. Highest Rated Dishes: Gruyere Omelette


Processing: user_data/processed_files/processed_651400dd5f57cba5d84a8d7d.json
Query for user_data/processed_files/processed_651400dd5f57cba5d84a8d7d.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Crudites & Sour Cream Dip , Lemon Quinoa with Roasted Protein, Mushroom Penne, Creamy Tomato Tagliatelle , Kalamata Olives & Parmesan Cheese, Sticky Mango & Coconut Rice Pudding 


Processing: user_data/processed_files/processed_66a3bcf4430fb89a1a6da0ab.json
Query for user_data/processed_files/processed_66a3bcf4430fb89a1a6da0ab.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Beetroot & Quinoa Salad , Tuscan Protein & Black Rice, Beef Stew with Sweet Potato Mash, Mushroom Meatballs & Mash, Bell Pepper Pesto Pasta , Coconut Curry and Quinoa 


Processing: user_data/processed_files/processed_64dde7c55d497c8a38be700d.json
Query for user_data/processed_files/processed_64dde7c55d497c8a38be700d.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Lemon Quinoa with Roasted Protein, Eggplant Lasagne, Beef Bourguignon, Ginger-Sesame Protein with Quinoa, Tomato Bulgur Pilaf, Machboos Rubyan


Processing: user_data/processed_files/processed_678b4379c954b8182666bfb7.json
Query for user_data/processed_files/processed_678b4379c954b8182666bfb7.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Southwest Tacos, Chimichurri Steak, Mushroom Penne, Coconut Curry and Quinoa, Shawarma Bowl , Satay & Jasmin Rice. Highest Rated Dishes: Gyro Bowl , Satay & Jasmin Rice, Ginger-Sesame Protein with Quinoa, Creamy Tomato Tagliatelle , Asian Rice Noodles


Processing: user_data/processed_files/processed_67e91b922e41aa516fa720c8.json
Query for user_data/processed_files/processed_67e91b922e41aa516fa720c8.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Beef Stew with Sweet Potato Mash, Maqluba , Cajun Chicken Bowl, Tuscan Protein & Black Rice, Pumpkin & Yoghurt Bowl, Med Breakfast Mezze


Processing: user_data/processed_files/processed_67ed63862e41aa516ffd93f1.json
Query for user_data/processed_files/processed_67ed63862e41aa516ffd93f1.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Creamy Tomato Zoodles , Beef Stew with Sweet Potato Mash, Beetroot & Quinoa Salad , Butter Chickpeas with Cauli Rice, Cajun Chicken Bowl, Bell Pepper Pesto Pasta 


Processing: user_data/processed_files/processed_651437cfd838112c9853c1f1.json
Query for user_data/processed_files/processed_651437cfd838112c9853c1f1.json:
Spice Level: Low, Cuisine: European. Popular Dishes: Butter Chickpeas with Cauli Rice, Mushroom Penne, Mini Quiches, Mushroom Spinach & Cheddar Omelette, Tomato Basil Soup, Cajun Chicken Bowl


Processing: user_data/processed_files/processed_679e294a9c40d6abe11128a5.json
Query for user_data/processed_files/processed_679e294a9c40d6abe11128a5.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mac & Cheese with Cauliflower, Reuben Sandwich, Eggplant Lasagne, Cajun Roasted Chicken Sandwich, Creamy Protein & Mash Potatoes, Chicken Tenders. Highest Rated Dishes: Lamb Stew with Colcannon Sweet Potato , Shawarma Bowl , Southwest Tacos, Musakhan & Vermicelli Rice, Tuscan Beans & Roasted Beef


Processing: user_data/processed_files/processed_64ea0ff65d497c8a38e10c6c.json
Query for user_data/processed_files/processed_64ea0ff65d497c8a38e10c6c.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Chicken a la King & Roasted Sweet Potato, Pumpkin & Yoghurt Bowl, Burrito Bowl , Leek & Potato Mash, Balkan Mushroom Rice, Mushroom Spinach & Cheddar Omelette


Processing: user_data/processed_files/processed_679cb995ed9daf35d79a116c.json
Query for user_data/processed_files/processed_679cb995ed9daf35d79a116c.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Mushroom Spinach & Cheddar Omelette, Gruyere Omelette, French Toast with Strawberry Compote, Creamy Tomato Zoodles , Kale Salad with Maple Tahina Dressing, Cheesy Omelette with Broccoli


Processing: user_data/processed_files/processed_64ff5cafb78e129edca5bc95.json
Query for user_data/processed_files/processed_64ff5cafb78e129edca5bc95.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Classic Chicken Salad, Choco Walnut , Omega Egg Protein Pot , Cheese and Nuts Pot, Mushroom Soup , Roasted Pecans


Processing: user_data/processed_files/processed_6735cc62e968b82d2e6464c1.json
Query for user_data/processed_files/processed_6735cc62e968b82d2e6464c1.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Greek Yogurt, Mushroom Penne, Berry Smoothie , Crudites & Sour Cream Dip , Bircher Muesli with Nuts & Berries., Ricotta Cheese Blueberry Pancake. Highest Rated Dishes: Choco Walnut , Ricotta Cheese Blueberry Pancake, Sticky Mango & Coconut Rice , Butter Masala & Rice , Oats Porridge 


Processing: user_data/processed_files/processed_65acebb35e35c6601bd97ea5.json
Query for user_data/processed_files/processed_65acebb35e35c6601bd97ea5.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Bell Pepper Pesto Pasta, Mushroom Penne, Fusilli Alfredo, Maqluba , Mansaf , Tagliatelle Ragu . Highest Rated Dishes: Spaghetti Bolognese, Leek & Potato Fusilli Pasta, Fusilli Alfredo, Eggplant Lasagne, Butter Masala & Rice 


Processing: user_data/processed_files/processed_67d7bd56fbc0be74eecf1736.json
Query for user_data/processed_files/processed_67d7bd56fbc0be74eecf1736.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Fusilli Alfredo, Chimichurri Steak, Herb Mashed Potato, Baked Protein & Mashed Potato, Creamy Tomato Tagliatelle , Beef Bourguignon. Highest Rated Dishes: Coconut Curry and Quinoa, Souvlaki & Quinoa Pilaf


Processing: user_data/processed_files/processed_64b39840e7008bc8ad39e52f.json
Query for user_data/processed_files/processed_64b39840e7008bc8ad39e52f.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Gruyere Omelette, English Breakfast, Breakfast Power Bowl , Shakshouka , Med Breakfast Mezze, Reuben Sandwich


Processing: user_data/processed_files/processed_656f60cef139eab3825448fa.json
Query for user_data/processed_files/processed_656f60cef139eab3825448fa.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mushroom Penne, Chicken Crunchy Bowl, Classic Chicken Salad , Mansaf , Maqluba , Senegalese Protein & Tomato Rice


Processing: user_data/processed_files/processed_677a91bcd77f7aa7b009df89.json
Query for user_data/processed_files/processed_677a91bcd77f7aa7b009df89.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Chicken Patty with Egg, Bell Pepper Pesto Pasta , Peruvian Chicken & Mash Potato, Tomato Bulgur Pilaf, Breakfast Power Bowl , Blackened Protein with Braised Lentils


Processing: user_data/processed_files/processed_674c4dadeb5b64b4b50e9ac1.json
Query for user_data/processed_files/processed_674c4dadeb5b64b4b50e9ac1.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Butter Chickpeas with Cauli Rice, Bell Pepper Pesto Pasta , Beef Stew with Sweet Potato Mash, Poke Bowl, Mushroom Penne, Maqluba . Highest Rated Dishes: Creamy Protein & Mash Potatoes, Maqluba 


Processing: user_data/processed_files/processed_67d2d1531d73f3185b8f1829.json
Query for user_data/processed_files/processed_67d2d1531d73f3185b8f1829.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Noodles Bolognese , Beef Stew with Sweet Potato Mash, Chimichurri Steak, Thai Mango Salad, Greek Chicken with Potato Wedges, Creamy Quinoa Bowl . Highest Rated Dishes: Beetroot & Quinoa Salad , Beef Stew with Sweet Potato Mash, Baked Protein & Mashed Potato, Butter Masala & Quinoa, Mushroom Meatballs & Mash


Processing: user_data/processed_files/processed_67abf9cdc8b635d2cccc5925.json
Query for user_data/processed_files/processed_67abf9cdc8b635d2cccc5925.json:
Spice Level: Medium, Cuisine: Arabic, European, Mediterranean. Popular Dishes: Peruvian Chicken & Mash Potato, Gyro Bowl , Blackened Protein with Braised Lentils, Maftoul with Zucchini & Minced Protein , Shawarma Bowl , Satay & Jasmin Rice


Processing: user_data/processed_files/processed_670214c6fe9082943a279142.json
Query for user_data/processed_files/processed_670214c6fe9082943a279142.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Cajun Roasted Chicken Sandwich, Creamy Tomato Tagliatelle , Waffles with Apple Compote, Pumpkin & Yoghurt Bowl , Cilbir, Tuscan Protein & Black Rice


Processing: user_data/processed_files/processed_6559b38ce79717090a73c533.json
Query for user_data/processed_files/processed_6559b38ce79717090a73c533.json:
Spice Level: Low, Cuisine: European. Popular Dishes: Oatmeal Apple Pancake, Chicken Tenders, Waffles with Apple Compote, Creamy Tomato Tagliatelle , Coconut & Flax Pancake, Mushroom Penne


Processing: user_data/processed_files/processed_6772da83af7b2389de1ddc57.json
Query for user_data/processed_files/processed_6772da83af7b2389de1ddc57.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mac & Cheese with Cauliflower, Blackened Protein with Braised Lentils, Gyro Bowl , Zoodles Ragu , Peri Peri Protein, Bell Pepper Pesto Pasta 


Processing: user_data/processed_files/processed_67795b36b9b192b285477df9.json
Query for user_data/processed_files/processed_67795b36b9b192b285477df9.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Chimichurri Steak, Mushroom Meatballs & Mash, Caramelized Chicken Sandwich, Mushroom Penne, Peruvian Chicken & Mash Potato, Peri Peri Protein


Processing: user_data/processed_files/processed_67f266d192687bdcfc76d7ec.json
Query for user_data/processed_files/processed_67f266d192687bdcfc76d7ec.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Cajun Roasted Chicken Sandwich, Bell Pepper Pesto Pasta, Waffles with Apple Compote, Peruvian Chicken & Mash Potato, Machboos Rubyan, Cilbir


Processing: user_data/processed_files/processed_66d55f181fcb0fc959a83699.json
Query for user_data/processed_files/processed_66d55f181fcb0fc959a83699.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Greek Salad, Maqluba , Creamy Quinoa Bowl , Chimichurri Steak, Tikka Protein with Saffron Rice, Kale & Quinoa Salad 


Processing: user_data/processed_files/processed_6512d21d5f57cba5d84682b6.json
Query for user_data/processed_files/processed_6512d21d5f57cba5d84682b6.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mac & Cheese with Cauliflower, Fusilli Alfredo, Creamy Protein & Mash Potatoes, Eggplant Lasagne, Bell Pepper Pesto Pasta, Balkan Mushroom Rice. Highest Rated Dishes: Asian Rice Noodles, Creamy Tomato Tagliatelle , Pumpkin & Yoghurt Bowl , Butter Chickpeas with Rice


Processing: user_data/processed_files/processed_6597f5f893f2206fb148f076.json
Query for user_data/processed_files/processed_6597f5f893f2206fb148f076.json:
Spice Level: Medium, Cuisine: American. Popular Dishes: Cajun Roasted Chicken Sandwich, Korean Style Wrap, Chimichurri Steak, Caramelized Chicken Sandwich, Peri Peri Wrap, Mansaf . Highest Rated Dishes: Chili Con Carne, Shawarma Bowl , Satay & Jasmin Rice, Lamb Stew with Saffron Rice, Coconut Curry and Quinoa


Processing: user_data/processed_files/processed_67a5e707c3ffb3a5a79167b0.json
Query for user_data/processed_files/processed_67a5e707c3ffb3a5a79167b0.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Peanut Banana Smoothie , Creamy Tomato Tagliatelle , Peri Peri Protein, Tomato Basil Soup, Acai Smoothie, Kalamata Olives & Parmesan Cheese


Processing: user_data/processed_files/processed_673dcb9c8135a8501bf138f0.json
Query for user_data/processed_files/processed_673dcb9c8135a8501bf138f0.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mac & Cheese with Cauliflower, Creamy Protein & Mash Potatoes, Mushroom Meatballs & Mash, Chimichurri Steak, Alfredo Mash, Leek & Potato Mash


Processing: user_data/processed_files/processed_64ca28ec2fb782b603462407.json
Query for user_data/processed_files/processed_64ca28ec2fb782b603462407.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Med Breakfast Mezze, Cajun Roasted Chicken Sandwich, Pumpkin & Yoghurt Bowl , Greek Salad, Breakfast Power Bowl , Cilbir


Processing: user_data/processed_files/processed_67cd362121dbdf828a37c1d4.json
Query for user_data/processed_files/processed_67cd362121dbdf828a37c1d4.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Butter Chickpeas with Cauli Rice, Cajun Chicken Bowl, Creamy Tomato Zoodles , Greek Salad , Pumpkin & Yoghurt Bowl, Thai Mango Salad. Highest Rated Dishes: Asian Zoodles, Butter Chickpeas with Cauli Rice, Herb Mash Potato & Mushroom Sauce , Cajun Chicken Bowl


Processing: user_data/processed_files/processed_6783ccfbea9ef8d9ee2c9ca2.json
Query for user_data/processed_files/processed_6783ccfbea9ef8d9ee2c9ca2.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Leek & Potato Mash, Poke Bowl, Gyro Bowl , Cajun Chicken Bowl, Herb Mash Potato & Mushroom Sauce , Pumpkin & Yoghurt Bowl. Highest Rated Dishes: Tuscan Beans & Roasted Beef, Hoisin Protein with Coconut Noodles, Mansaf, Lamb Stew with Sweet Potato Mash, Senegalese Protein & Cauli Rice


Processing: user_data/processed_files/processed_64c40e6c2fb782b60334d930.json
Query for user_data/processed_files/processed_64c40e6c2fb782b60334d930.json:
Spice Level: Medium, Cuisine: European, Mediterranean. Popular Dishes: Beef Stew with Saffron Rice, Creamy Tomato Tagliatelle , Peruvian Chicken & Mash Potato, Mushroom Meatballs & Mash, Tikka Protein with Saffron Rice, Eggplant Lasagne


Processing: user_data/processed_files/processed_67c892ff8a4937be42cb14d0.json
Query for user_data/processed_files/processed_67c892ff8a4937be42cb14d0.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Bell Pepper Pesto Pasta , Beetroot & Quinoa Salad , Pumpkin & Yoghurt Bowl, Coconut Curry and Quinoa , Kale & Quinoa Salad , Mac & Cheese with Cauliflower


Processing: user_data/processed_files/processed_6731fa2853e197222dee46de.json
Query for user_data/processed_files/processed_6731fa2853e197222dee46de.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Breakfast Power Bowl , English Breakfast, Fried Eggs with Cheesy Cauliflower, Chimichurri Steak, Shakshouka , Mushroom Meatballs & Mash


Processing: user_data/processed_files/processed_67a27178417227f487135365.json
Query for user_data/processed_files/processed_67a27178417227f487135365.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Kale Salad with Maple Tahina Dressing, Bell Pepper Pesto Pasta , Musakhan & Cauli Rice, Mansaf, Blackened Protein with Braised Lentils, Butter Masala & Quinoa


Processing: user_data/processed_files/processed_64a7adf4144bc0a5eb914149.json
Query for user_data/processed_files/processed_64a7adf4144bc0a5eb914149.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Mushroom Soup , Sticky Mango & Coconut Rice Pudding , Tomato Basil Soup, Coconut Curry and Quinoa , Broccoli Cheese Soup, Eggplant Lasagne 


Processing: user_data/processed_files/processed_67441ac5872a8b3396595cdf.json
Query for user_data/processed_files/processed_67441ac5872a8b3396595cdf.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Herb Mashed Potato, Sumac Protein & Lemon Herb Couscous , Maqluba , Senegalese Protein & Tomato Rice, Tuscan Protein & Black Rice, Shawarma Bowl 


Processing: user_data/processed_files/processed_652eafd7a186fe615e48d827.json
Query for user_data/processed_files/processed_652eafd7a186fe615e48d827.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Fusilli Alfredo, Mushroom Penne, Asian Rice Noodles, Spaghetti Bolognese, Leek & Potato Fusilli Pasta, Caramelized Chicken Sandwich


Processing: user_data/processed_files/processed_65662d137da596aac2f833c0.json
Query for user_data/processed_files/processed_65662d137da596aac2f833c0.json:
Spice Level: Low, Cuisine: European, Mediterranean. Popular Dishes: Butter Chickpeas with Cauli Rice, Mushroom Penne, Mushroom Meatballs & Mash, Peruvian Chicken & Mash Potato, Cajun Chicken Bowl, Creamy Tomato Zoodles 


Processing: user_data/processed_files/processed_67b0ca24bb439e10a7e42e23.json
Query for user_data/processed_files/processed_67b0ca24bb439e10a7e42e23.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Kunafa Cup, Chocolate Raspberry Mousse, Classic Chicken Salad , Beef Stew with Saffron Rice, Berry Smoothie , Mansaf 


Processing: user_data/processed_files/processed_66fa0a200dd09de274b188ff.json
Query for user_data/processed_files/processed_66fa0a200dd09de274b188ff.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Creamy Tomato Zoodles , Cajun Chicken Bowl, Beetroot & Quinoa Salad , Orzo Salad & Maple Lemon Dressing, Korean Protein Bowl, Chimichurri Steak


Processing: user_data/processed_files/processed_6756b926b7435fe34b0986cf.json
Query for user_data/processed_files/processed_6756b926b7435fe34b0986cf.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Peri Peri Wrap, Coconut Curry and Quinoa, Butter Chickpeas with Rice, Musakhan & Vermicelli Rice, Creamy Protein & Mash Potatoes, Tuscan Protein & Black Rice


Processing: user_data/processed_files/processed_6777fbda850445cceb13f8e4.json
Query for user_data/processed_files/processed_6777fbda850445cceb13f8e4.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Crudites & Sour Cream Dip, Beef Stew with Sweet Potato Mash, Bell Pepper Pesto Pasta , Asian Zoodles, Tuna Nicoise Salad, Beetroot & Quinoa Salad . Highest Rated Dishes: Mediterranean Salad, Ginger-Sesame Protein with Quinoa, Zoodles Ragu , Thai Mango Salad, Orzo Salad & Maple Lemon Dressing


Processing: user_data/processed_files/processed_67f0388192687bdcfc52e4eb.json
Query for user_data/processed_files/processed_67f0388192687bdcfc52e4eb.json:
Spice Level: Medium, Cuisine: Arabic, Mediterranean. Popular Dishes: Pumpkin & Yoghurt Bowl , Creamy Tomato Tagliatelle , Southwest Tacos, Mushroom Penne, Tomato Bulgur Pilaf, Butter Chickpeas with Rice


Processing: user_data/processed_files/processed_66d099c5d877ae126aa1d873.json
Query for user_data/processed_files/processed_66d099c5d877ae126aa1d873.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Ginger-Sesame Protein with Quinoa, Lamb Stew with Colcannon Sweet Potato , Lamb Stew with Saffron Rice, Leek & Potato Fusilli Pasta, Maftoul with Zucchini & Minced Protein , Pumpkin & Yoghurt Bowl . Highest Rated Dishes: Maqluba 


Processing: user_data/processed_files/processed_67a8d3f0c3ffb3a5a7ceef7a.json
Query for user_data/processed_files/processed_67a8d3f0c3ffb3a5a7ceef7a.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mushroom Meatballs & Mash, Eggplant Lasagne, Peruvian Chicken & Mash Potato, Chimichurri Steak, Mushroom Penne, Blackened Protein with Braised Lentils. Highest Rated Dishes: Blackened Protein with Braised Lentils


Processing: user_data/processed_files/processed_67cec8da4efbabe001ceca58.json
Query for user_data/processed_files/processed_67cec8da4efbabe001ceca58.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Cheese and Nuts Pot, Kalamata Olives & Parmesan Cheese, Grapes Pot, Chimichurri Steak, Burrito Bowl, Leek & Potato Fusilli Pasta. Highest Rated Dishes: Penne Napolitaine , Gyro Bowl , Sticky Toffee Pudding, Maqluba , Maftoul with Zucchini & Minced Protein 


Processing: user_data/processed_files/processed_6772882daf7b2389de0d7718.json
Query for user_data/processed_files/processed_6772882daf7b2389de0d7718.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Classic Chicken Salad, Omega Egg Protein Pot , Omega Egg with Parmesan & Pesto, Almond Brownie , Tuna Salad, Chicken Crunchy Bowl. Highest Rated Dishes: Chicken Biryani Bowl , Almond Brownie , Mushroom Soup , Truffle & Chive Scones , Tuna Salad


Processing: user_data/processed_files/processed_66838c6258a19bedb59b3d83.json
Query for user_data/processed_files/processed_66838c6258a19bedb59b3d83.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mushroom Meatballs & Mash, Peruvian Chicken & Mash Potato, Tuscan Protein & Barley, Pumpkin & Yoghurt Bowl , Burrito Bowl, Eggplant Lasagne


Processing: user_data/processed_files/processed_679b1ba1b32c7b51e771fedf.json
Query for user_data/processed_files/processed_679b1ba1b32c7b51e771fedf.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mediterranean Salad, Ginger-Sesame Protein with Quinoa, Herb Mashed Potato, Cowboy Salad , Orzo Salad & Maple Lemon Dressing, Caramelized Chicken Sandwich


Processing: user_data/processed_files/processed_66d1a8014031ac639fb91716.json
Query for user_data/processed_files/processed_66d1a8014031ac639fb91716.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Tikka Protein with Roasted Veggies, Chicken Tenders, Creamy Tomato Zoodles , Breakfast Power Bowl , Mushroom Penne, Chicken Patty with Egg


Processing: user_data/processed_files/processed_6798c5d425b2b4fb837c171c.json
Query for user_data/processed_files/processed_6798c5d425b2b4fb837c171c.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Sticky Mango & Coconut Rice , Tuna Salad, Fusilli Alfredo, Cheese and Nuts Pot, Berry Smoothie , Cajun Chicken Bowl. Highest Rated Dishes: Carrot & Pineapple Ginger Juice, Mini Quiches, Mushroom Meatballs & Mash, Coconut Choco Overnight Oats , Berry Smoothie 


Processing: user_data/processed_files/processed_64b18598ece526dd8cbc5675.json
Query for user_data/processed_files/processed_64b18598ece526dd8cbc5675.json:
Spice Level: Medium, Cuisine: Arabic, Fusion, Mediterranean. Popular Dishes: Peruvian Chicken & Mash Potato, Tomato Bulgur Pilaf, Breakfast Power Bowl 


Processing: user_data/processed_files/processed_6783d2f5ea9ef8d9ee2d9746.json
Query for user_data/processed_files/processed_6783d2f5ea9ef8d9ee2d9746.json:
Spice Level: Medium, Cuisine: Arabic, Mediterranean. Popular Dishes: Maqluba , Butternut Squash with Orzo, Mujaddara , Butter Chickpeas with Rice, Peri Peri Protein, Chimichurri Steak


Processing: user_data/processed_files/processed_67c45c88776c9d06c3f03beb.json
Query for user_data/processed_files/processed_67c45c88776c9d06c3f03beb.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Chicken Patty with Egg, Pumpkin & Yoghurt Bowl, Cilbir, Breakfast Power Bowl , Mushroom Meatballs & Mash, Chicken Tenders. Highest Rated Dishes: Tikka Protein with Roasted Veggies, Chicken Tenders


Processing: user_data/processed_files/processed_64e385885d497c8a38cb0ffd.json
Query for user_data/processed_files/processed_64e385885d497c8a38cb0ffd.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: French Toast with Strawberry Compote, Bell Pepper Pesto Pasta , Musakhan & Cauli Rice, Yoghurt Granola , Beef Stew with Sweet Potato Mash, Kunafa Cup. Highest Rated Dishes: Oats Porridge 


Processing: user_data/processed_files/processed_672a1cd3b72ef295e1b765ac.json
Query for user_data/processed_files/processed_672a1cd3b72ef295e1b765ac.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Peruvian Chicken & Mash Potato, Musakhan & Cauli Rice, Butter Masala & Quinoa, Balkan Mushroom Rice, Tomato Bulgur Pilaf, Mushroom Penne. Highest Rated Dishes: Butter Masala & Quinoa, Herb Mashed Potato


Processing: user_data/processed_files/processed_67e0f8d7a578b5f9054f9ef7.json
Query for user_data/processed_files/processed_67e0f8d7a578b5f9054f9ef7.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Beef Bourguignon , Sumac Protein & Roasted Veggies, Cajun Chicken Bowl, Kale Salad with Maple Tahina Dressing, Baked Protein & Mashed Potato, Burrito Bowl . Highest Rated Dishes: Beef Bourguignon , Chimichurri Steak, Creamy Protein & Mash Potatoes, Chicken a la King & Roasted Sweet Potato


Processing: user_data/processed_files/processed_6767a2c65ea91f54c84866bc.json
Query for user_data/processed_files/processed_6767a2c65ea91f54c84866bc.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Bell Pepper Pesto Pasta , Cajun Chicken Bowl, Beetroot & Quinoa Salad , Creamy Protein & Mash Potatoes, Mansaf, Creamy Quinoa Bowl . Highest Rated Dishes: Herb Mash Potato & Mushroom Sauce , Cajun Chicken Bowl, Pumpkin & Yoghurt Bowl, Beetroot & Quinoa Salad , Tikka Protein with Roasted Veggies


Processing: user_data/processed_files/processed_65958e18f2f6720a58585071.json
Query for user_data/processed_files/processed_65958e18f2f6720a58585071.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Pumpkin & Yoghurt Bowl , Chocolate Muffin , Bell Pepper Pesto Pasta, Raspberry Yoghurt , Chocolate Raspberry Mousse, Crudites & Sour Cream Dip 


Processing: user_data/processed_files/processed_64e208c25d497c8a38c73dae.json
Query for user_data/processed_files/processed_64e208c25d497c8a38c73dae.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Caramelized Chicken Sandwich, Cajun Roasted Chicken Sandwich, Butter Masala & Rice , Creamy Tomato Tagliatelle , Creamy Protein & Mash Potatoes, Chicken Biryani Bowl . Highest Rated Dishes: Eggplant Lasagne, Butter Masala & Rice , Lime Chili Quesadillas 


Processing: user_data/processed_files/processed_6513dad94c207151f207710c.json
Query for user_data/processed_files/processed_6513dad94c207151f207710c.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Cilbir, Coconut Curry and Quinoa , Mfarakeh, Butter Chickpeas with Cauli Rice, Gyro Bowl , Breakfast Power Bowl 


Processing: user_data/processed_files/processed_66f84965b2549e11fc2c4351.json
Query for user_data/processed_files/processed_66f84965b2549e11fc2c4351.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Breakfast Power Bowl , Coconut Choco Overnight Oats , Chicken Patty with Egg, Waffles with Apple Compote, Med Breakfast Mezze, Chicken Tenders


Processing: user_data/processed_files/processed_659af0c214c28bbc7574d932.json
Query for user_data/processed_files/processed_659af0c214c28bbc7574d932.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Shawarma Bowl , Mac & Cheese with Cauliflower, Creamy Protein & Mash Potatoes, Creamy Quinoa Bowl , Baked Protein & Mashed Potato, Greek Chicken with Potato Wedges. Highest Rated Dishes: Asian Meatballs with Mash Potato


Processing: user_data/processed_files/processed_66dd5f8f69b80f640432f3cf.json
Query for user_data/processed_files/processed_66dd5f8f69b80f640432f3cf.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Creamy Tomato Tagliatelle , Mushroom Meatballs & Mash, Beef Bourguignon, Tikka Protein with Saffron Rice, Satay & Jasmin Rice, Beef Stew with Saffron Rice. Highest Rated Dishes: Creamy Tomato Tagliatelle , Orzo Salad & Maple Lemon Dressing, Satay & Jasmin Rice, Tagliatelle Ragu , Poke Bowl


Processing: user_data/processed_files/processed_66cd915be7a9ca87b0536ccc.json
Query for user_data/processed_files/processed_66cd915be7a9ca87b0536ccc.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Asian Meatballs with Fried Rice, Gyro Bowl , Bell Pepper Pesto Pasta, Bell Pepper Pesto Pasta , Souvlaki & Quinoa Pilaf, Greek Salad 


Processing: user_data/processed_files/processed_6652fec04df8c59ae54b5988.json
Query for user_data/processed_files/processed_6652fec04df8c59ae54b5988.json:
Spice Level: Medium, Cuisine: European, Fusion, Mediterranean. Popular Dishes: Cajun Roasted Chicken Sandwich, Creamy Tomato Tagliatelle , Butter Chickpeas with Rice, Mushroom Penne, Senegalese Protein & Tomato Rice, Mushroom Meatballs & Mash. Highest Rated Dishes: Cajun Roasted Chicken Sandwich, Creamy Tomato Tagliatelle 


Processing: user_data/processed_files/processed_66acddb36777e93279d8b6db.json
Query for user_data/processed_files/processed_66acddb36777e93279d8b6db.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mushroom Meatballs & Mash, Greek Salad, Poke Bowl, Sumac Protein & Lemon Herb Couscous , Peruvian Chicken & Mash Potato, Quinoa with Ratatouille 


Processing: user_data/processed_files/processed_64bbc31847e19d679e022a2c.json
Query for user_data/processed_files/processed_64bbc31847e19d679e022a2c.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Creamy Tomato Tagliatelle , Beef Stew with Saffron Rice, Coconut Curry and Quinoa, Chimichurri Steak, Creamy Quinoa Bowl , Mushroom Penne


Processing: user_data/processed_files/processed_677cc8fc7913934dc0872460.json
Query for user_data/processed_files/processed_677cc8fc7913934dc0872460.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Crudites & Sour Cream Dip, Classic Chicken Salad, Omega Egg Protein Pot , Eggplant Lasagne , Greek Salad , Tuna Nicoise Salad. Highest Rated Dishes: Crudites & Sour Cream Dip, Tomato Basil Soup, Crudites & Smoked Paprika Dip 


Processing: user_data/processed_files/processed_679cce6eed9daf35d79cb291.json
Query for user_data/processed_files/processed_679cce6eed9daf35d79cb291.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Greek Yogurt, Yoghurt Granola , Chocolate Yoghurt , Raspberry Fruit Bowl, Shakshouka , Baked Protein & Mashed Potato


Processing: user_data/processed_files/processed_65e15f9137c1c7d22c7a6edc.json
Query for user_data/processed_files/processed_65e15f9137c1c7d22c7a6edc.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Caramelized Chicken Sandwich, Cajun Roasted Chicken Sandwich, Bell Pepper Pesto Pasta, Mac & Cheese with Cauliflower, Mushroom Meatballs & Mash, Roasted Protein & Mash Potato. Highest Rated Dishes: Roasted Protein & Mash Potato, Poke Bowl


Processing: user_data/processed_files/processed_66dd932969b80f640434ffac.json
Query for user_data/processed_files/processed_66dd932969b80f640434ffac.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Tikka Protein with Saffron Rice, Blackened Protein with Braised Lentils, Coconut Curry and Quinoa, Lamb Stew with Colcannon Sweet Potato , Pumpkin & Yoghurt Bowl , Chimichurri Steak


Processing: user_data/processed_files/processed_64c7fc7d2fb782b6033f49e1.json
Query for user_data/processed_files/processed_64c7fc7d2fb782b6033f49e1.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Senegalese Protein & Tomato Rice, Peruvian Chicken & Mash Potato, Tikka Protein with Saffron Rice, Kale Salad with Maple Tahina Dressing, Fusilli Alfredo, Machboos Rubyan


Processing: user_data/processed_files/processed_67b2eb46bb439e10a70a1db8.json
Query for user_data/processed_files/processed_67b2eb46bb439e10a70a1db8.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Burrito Bowl, Creamy Tomato Tagliatelle , Cajun Roasted Chicken Sandwich, Chicken a la King & Onion Bread Roll, Shawarma Bowl , Beetroot & Quinoa Salad


Processing: user_data/processed_files/processed_64bc013fc3b0d43faabe6d56.json
Query for user_data/processed_files/processed_64bc013fc3b0d43faabe6d56.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Reuben Sandwich, Chicken Patty with Egg, Gruyere Omelette, Peruvian Chicken & Mash Potato, Lemon Roasted Chicken & Beet Avocado Bagel, Classic French Toast. Highest Rated Dishes: Fried Eggs with Turkey Bacon, Poke Bowl, English Breakfast, Chicken Patty with Egg, Reuben Sandwich


Processing: user_data/processed_files/processed_66dd511869b80f6404320338.json
Query for user_data/processed_files/processed_66dd511869b80f6404320338.json:
Spice Level: Low, Cuisine: Comfort Food. Popular Dishes: Ricotta Cheese Blueberry Pancake, Berry Smoothie , Butter Chickpeas with Rice, Roasted Pecans, Senegalese Protein & Tomato Rice, Med Breakfast Mezze. Highest Rated Dishes: Butter Chickpeas with Rice, Ricotta Cheese Blueberry Pancake, Sticky Mango & Coconut Rice , Panna Cotta with Raspberry, Crudites & Sour Cream Dip 


Processing: user_data/processed_files/processed_672a1c3fb72ef295e1b75f7c.json
Query for user_data/processed_files/processed_672a1c3fb72ef295e1b75f7c.json:
Spice Level: Medium, Cuisine: Arabic. Popular Dishes: Quinoa Biryani, Bell Pepper Pesto Pasta , Cajun Chicken Bowl, Tomato Bulgur Pilaf, Senegalese Protein & Roasted Veggies, Mushroom Meatballs & Mash


Processing: user_data/processed_files/processed_666c1a3d55e8c7db1b17e8fd.json
Query for user_data/processed_files/processed_666c1a3d55e8c7db1b17e8fd.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Peri Peri Protein, Sumac Protein & Roasted Veggies, Beef Stew with Sweet Potato Mash, Mansaf, Souvlaki & Quinoa Pilaf, Ginger-Sesame Protein with Quinoa


Processing: user_data/processed_files/processed_67bd7055c0200de98748599a.json
Query for user_data/processed_files/processed_67bd7055c0200de98748599a.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Bell Pepper Pesto Pasta , Burrito Bowl , Tikka Protein with Roasted Veggies, Cajun Chicken Bowl, Greek Chicken with Potato Wedges, Beetroot & Quinoa Salad . Highest Rated Dishes: Bell Pepper Pesto Pasta , Greek Chicken with Potato Wedges, Coconut Curry and Quinoa , Korean Protein Bowl, Tikka Protein with Roasted Veggies


Processing: user_data/processed_files/processed_661a5e7b566a4aef140db162.json
Query for user_data/processed_files/processed_661a5e7b566a4aef140db162.json:
Spice Level: Medium, Cuisine: American. Popular Dishes: Chimichurri Steak, Cajun Roasted Chicken Sandwich, Southwest Tacos, Beef Stew with Saffron Rice, Tuscan Protein & Black Rice, Mushroom Meatballs & Mash. Highest Rated Dishes: Chimichurri Steak


Processing: user_data/processed_files/processed_64f48df85d497c8a3806286c.json
Query for user_data/processed_files/processed_64f48df85d497c8a3806286c.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Fusilli Alfredo, Creamy Tomato Tagliatelle , Mushroom Meatballs & Mash, Balkan Mushroom Rice, Creamy Quinoa Bowl , Asian Rice Noodles


Processing: user_data/processed_files/processed_66a9aac451fcd02361eb4433.json
Query for user_data/processed_files/processed_66a9aac451fcd02361eb4433.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Southwest Tacos, Bell Pepper Pesto Pasta, Coconut Curry and Quinoa, Satay & Jasmin Rice, Creamy Quinoa Bowl , Butter Masala & Rice 


Processing: user_data/processed_files/processed_651297a7c4b25f5cc11d1ee6.json
Query for user_data/processed_files/processed_651297a7c4b25f5cc11d1ee6.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Beef Bourguignon, Machboos Rubyan, Breakfast Power Bowl , Chicken Tenders, Chimichurri Steak, Dates & Mango Bircher Muesli 


Processing: user_data/processed_files/processed_64c64440cf697b0d785607ce.json
Query for user_data/processed_files/processed_64c64440cf697b0d785607ce.json:
Spice Level: Medium, Cuisine: Asian, European. Popular Dishes: Butter Chickpeas with Rice, Creamy Tomato Tagliatelle , Mushroom Penne, Mushroom Meatballs & Mash, Cajun Roasted Chicken Sandwich, Pumpkin & Yoghurt Bowl 


Processing: user_data/processed_files/processed_6624a5e41b57fe5ac6dab45a.json
Query for user_data/processed_files/processed_6624a5e41b57fe5ac6dab45a.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Mushroom Penne, Chimichurri Steak, Greek Chicken with Herb Rice, Lamb Stew with Colcannon Sweet Potato , Eggplant Lasagne, Satay & Jasmin Rice. Highest Rated Dishes: Butter Masala & Rice , Chimichurri Steak, Roasted Protein & Mash Potato, Fusilli Alfredo, Eggplant Lasagne


Processing: user_data/processed_files/processed_652503989ededaea3b1a11fc.json
Query for user_data/processed_files/processed_652503989ededaea3b1a11fc.json:
Spice Level: Medium, Cuisine: Arabic, European, Fusion. Popular Dishes: Creamy Tomato Zoodles , Cajun Chicken Bowl, Tomato Bulgur Pilaf, Quinoa Biryani, Tuscan Protein & Black Rice, Beef Stew with Sweet Potato Mash. Highest Rated Dishes: Butter Chickpeas with Cauli Rice


Processing: user_data/processed_files/processed_6517dec03f594baadf1fbbaf.json
Query for user_data/processed_files/processed_6517dec03f594baadf1fbbaf.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Beef Bourguignon, Cowboy Salad , Beetroot & Quinoa Salad, Lamb Stew with Colcannon Sweet Potato , Shawarma Bowl , Creamy Protein & Mash Potatoes


Processing: user_data/processed_files/processed_66edb99aa7bd87089756ec80.json
Query for user_data/processed_files/processed_66edb99aa7bd87089756ec80.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Caramelized Chicken Sandwich, Cajun Roasted Chicken Sandwich, Southwest Tacos, Chimichurri Steak, Mushroom Meatballs & Mash, Peri Peri Wrap. Highest Rated Dishes: Caramelized Chicken Sandwich


Processing: user_data/processed_files/processed_67c9a65221dbdf828a086373.json
Query for user_data/processed_files/processed_67c9a65221dbdf828a086373.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Peri Peri Protein, Chili Con Carne, Sumac Protein & Lemon Herb Couscous , Quinoa with Ratatouille , Southwest Tacos, Lamb Stew with Colcannon Sweet Potato 


Processing: user_data/processed_files/processed_67061ae623ead763336b938f.json
Query for user_data/processed_files/processed_67061ae623ead763336b938f.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Cajun Chicken Bowl, Senegalese Protein & Roasted Veggies, Butter Chickpeas with Cauli Rice, Creamy Quinoa Bowl , Beef Bourguignon , Peruvian Chicken & Mash Potato. Highest Rated Dishes: Creamy Quinoa Bowl , Beef Bourguignon , Cajun Chicken Bowl, Peruvian Chicken & Mash Potato, Bell Pepper Pesto Pasta 


Processing: user_data/processed_files/processed_668a6238b93a8181c31182e5.json
Query for user_data/processed_files/processed_668a6238b93a8181c31182e5.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Mushroom Spinach & Cheddar Omelette, Double Fried Eggs with Sausage Hash , Satay & Jasmin Rice, Reuben Sandwich, Cajun Roasted Chicken Sandwich, Mushroom Penne. Highest Rated Dishes: Double Fried Eggs with Sausage Hash , Eggplant Lasagne, Lime Chili Quesadillas , Chili Con Carne


Processing: user_data/processed_files/processed_657ef2623edd2150baa1a370.json
Query for user_data/processed_files/processed_657ef2623edd2150baa1a370.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Caramelized Chicken Sandwich, Poke Bowl, Chimichurri Steak, Tagliatelle Ragu , Tikka Protein with Saffron Rice, Creamy Tomato Tagliatelle 


Processing: user_data/processed_files/processed_66adbe4d97a511eeec9863c7.json
Query for user_data/processed_files/processed_66adbe4d97a511eeec9863c7.json:
Spice Level: Low, Cuisine: Mediterranean. Popular Dishes: Creamy Protein & Mash Potatoes, Pumpkin & Yoghurt Bowl, Baked Protein & Mashed Potato, Dates & Almond Smoothie , Beef Bourguignon , Lamb Stew with Sweet Potato Mash. Highest Rated Dishes: Creamy Protein & Mash Potatoes, Dates & Almond Smoothie , Alfredo Mash, Herb Mash Potato & Mushroom Sauce , Pumpkin & Yoghurt Bowl


Processing: user_data/processed_files/processed_676c4e32eac0c7b22a7734c2.json
Query for user_data/processed_files/processed_676c4e32eac0c7b22a7734c2.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: English Loaf, Mac & Cheese with Cauliflower, Peri Peri Protein, Blackened Protein with Braised Lentils, Ginger-Sesame Protein with Quinoa, Sumac Protein & Lemon Herb Couscous 


Processing: user_data/processed_files/processed_64a70cc5144bc0a5eb90cc11.json
Query for user_data/processed_files/processed_64a70cc5144bc0a5eb90cc11.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Senegalese Protein & Tomato Rice, Mushroom Meatballs & Mash, Burrito Bowl, Machboos Rubyan, Bell Pepper Pesto Pasta, Mansaf 


Processing: user_data/processed_files/processed_6720a6a123fc3d86743881bf.json
Query for user_data/processed_files/processed_6720a6a123fc3d86743881bf.json:
Spice Level: Medium, Cuisine: European. Popular Dishes: Eggplant Lasagne, Ricotta Cheese Blueberry Pancake, Tikka Protein with Saffron Rice, Chicken Casserole, English Breakfast, Herb Mashed Potato. Highest Rated Dishes: Fusilli Alfredo, Chili Con Carne, Shakshouka , Peri Peri Protein, Herb Mashed Potato

"""

# Step 1: Parse the multiline string to extract user_id and query
user_queries = {}

lines = multiline_string.strip().split('\n')
current_user = None

for line in lines:
    if line.startswith("Processing:"):
        # Extract user_id from filename (before .json)
        filename = line.split('/')[-1]
        current_user = filename.replace('processed_', '').replace('.json', '')
    elif line.startswith("Spice Level:"):
        if current_user:
            user_queries[current_user] = line.strip()

# Step 2: Update the CSV (replace with your actual CSV path)
csv_file_path = 'csv/merged_output.csv'  # Input CSV
output_csv_path = 'csv/llm_generated_ap4.csv'  # Output CSV

with open(csv_file_path, 'r') as csv_input, open(output_csv_path, 'w', newline='') as csv_output:
    reader = csv.DictReader(csv_input)
    fieldnames = reader.fieldnames + ['query']  # Add new column
    writer = csv.DictWriter(csv_output, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        # If customer_id exists in user_queries, add the query, else leave empty
        row['query'] = user_queries.get(row['customer_id'], '')
        writer.writerow(row)

print(f"Updated CSV saved to {output_csv_path}")
print("Extracted queries for users:", list(user_queries.keys()))

Updated CSV saved to csv/llm_generated_ap4.csv
Extracted queries for users: ['67961709889e8bdf42039a1b', '651400dd5f57cba5d84a8d7d', '66a3bcf4430fb89a1a6da0ab', '64dde7c55d497c8a38be700d', '678b4379c954b8182666bfb7', '67e91b922e41aa516fa720c8', '67ed63862e41aa516ffd93f1', '651437cfd838112c9853c1f1', '679e294a9c40d6abe11128a5', '64ea0ff65d497c8a38e10c6c', '679cb995ed9daf35d79a116c', '64ff5cafb78e129edca5bc95', '6735cc62e968b82d2e6464c1', '65acebb35e35c6601bd97ea5', '67d7bd56fbc0be74eecf1736', '64b39840e7008bc8ad39e52f', '656f60cef139eab3825448fa', '677a91bcd77f7aa7b009df89', '674c4dadeb5b64b4b50e9ac1', '67d2d1531d73f3185b8f1829', '67abf9cdc8b635d2cccc5925', '670214c6fe9082943a279142', '6559b38ce79717090a73c533', '6772da83af7b2389de1ddc57', '67795b36b9b192b285477df9', '67f266d192687bdcfc76d7ec', '66d55f181fcb0fc959a83699', '6512d21d5f57cba5d84682b6', '6597f5f893f2206fb148f076', '67a5e707c3ffb3a5a79167b0', '673dcb9c8135a8501bf138f0', '64ca28ec2fb782b603462407', '67cd362121dbdf828a37c1d4',